# CRISP-DM — Risque de panne par équipement

Ce notebook est limité à la classification du risque de panne. Les prévisions de charge technicien et de pièces PRC ont été retirées.

## 1. Business Understanding

Prédire si le nombre de curatifs du mois suivant dépassera la médiane historique de chaque équipement, afin de prioriser la maintenance.

In [1]:
# 2. Data Understanding et 3. Data Preparation
# La fonction build_dataset de retrain_model conserve toutes les variables utiles :
# lags 1-3, fenêtres glissantes 3 mois, MTBF, disponibilité, arrêts,
# ratio curatif/préventif, volumes et durées.
from retrain_model import get_engine, build_dataset

engine = get_engine()
interventions, training, scoring, features = build_dataset(engine)
print(f'Equipements : {interventions.equip_id.nunique()}')
print(f'Observations : {len(training)} | Features : {len(features)}')
training[['equip_id', 'annee_mois', 'curatif_mois_suivant']].head()


KeyboardInterrupt



In [ ]:
# 4. Modeling et 5. Evaluation
# Le script de production applique un split temporel 80/20, Gradient Boosting
# et une calibration isotonique; il journalise AUC train/test.
training = training.sort_values(['annee_mois', 'equip_id']).reset_index(drop=True)
split = int(len(training) * 0.8)
print(f'Train temporel : {split} | Test temporel : {len(training) - split}')
print('Cible : curatifs M+1 > médiane historique de l équipement')

In [ ]:
# 6. Deployment
# Lance le pipeline complet : champion/challenger (tolérance AUC 0.005),
# prédictions calibrées et catégories par percentiles 66 et 85.
from retrain_model import main
main()

# Sorties : best_model.pkl, predictions_risque.csv et model_meta.json.